# Fine-Tune Tutor Guardian — Arabic Parenting LLM
---
Fine-tunes a base model (Qwen2.5-3B/7B or Gemma-4B) on Arabic parenting Q&A data.
Output: a GGUF-quantized model ready for Ollama deployment.

**Prerequisites**: Upload your `qa_dataset.jsonl` file (generated by `ops/tools/generate_qa_dataset.py`)

**Runtime**: T4 GPU (~2-3h for 3B, ~20 min for 1.5B)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install -qU \
    transformers==4.47.1 \
    datasets==3.2.0 \
    trl==0.14.0 \
    peft==0.14.0 \
    accelerate==1.2.1 \
    bitsandbytes==0.45.2 \
    wandb \
    sentencepiece

import json
import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from google.colab import files

print("✅ All dependencies installed")

In [ ]:
# ── Upload QA dataset ──────────────────────────────────────────────────
print("📤 Upload qa_dataset.jsonl (generated from your knowledge base)")
uploaded = files.upload()

dataset_path = list(uploaded.keys())[0]
print(f"✅ Loaded: {dataset_path} ({os.path.getsize(dataset_path)/1024:.1f} KB)")

In [ ]:
# ── Load and format dataset ────────────────────────────────────────────
records = []
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"📊 Loaded {len(records)} Q&A pairs")

# Print sample
if records:
    print("\nSample:")
    print(f"  Instruction: {records[0]['instruction'][:80]}...")
    print(f"  Output: {records[0]['output'][:100]}...")
    print(f"  Domain: {records[0].get('domain', '-')}")

In [ ]:
# ── Model selection ────────────────────────────────────────────────────
# Options:
#   "Qwen/Qwen2.5-3B-Instruct"      — balanced quality/speed (~3h on T4)
#   "Qwen/Qwen2.5-1.5B-Instruct"    — fast (~45min), good for testing
#   "MorganaAI/Gemma-4B-Instruct"   — good Arabic, ~1.5h
#   "google/gemma-2-2b-it"          — fast (~1h)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # ← CHANGE THIS for larger model

print(f"📦 Base model: {MODEL_NAME}")

In [ ]:
# ── Load tokenizer ─────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Chat template for Qwen-style models
CHAT_TEMPLATE = """<|im_start|>system
أنت مساعد تربوي ذكي للأهل العرب المسلمين. أجب على آخر سؤال فقط. استخدم المعلومات المقدمة ولا تختلق.
<|im_end|>
{{#each messages}}
{{#if (eq role 'user')}}<|im_start|>user
{{content}}<|im_end|>
{{/if}}
{{#if (eq role 'assistant')}}<|im_start|>assistant
{{content}}<|im_end|>
{{/if}}
{{/each}}"""

# Format each record into the Qwen chat template
def format_chat(instruction, output):
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

formatted = [format_chat(r["instruction"], r["output"]) for r in records]

# Show a sample
print("\n📝 Formatted sample:")
print(formatted[0][:200] + "...")
print("\n" + "─" * 50)

# Create HF Dataset
dataset = Dataset.from_dict({"text": formatted})
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

In [ ]:
# ── LoRA configuration ─────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,  # rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"LoRA rank={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

In [ ]:
# ── Training arguments ─────────────────────────────────────────────────
output_dir = "./tg-finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",  # change to "wandb" for tracking
    push_to_hub=False,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    ddp_find_unused_parameters=False,
    gradient_checkpointing=True,
)

print(f"📋 Batch size: {training_args.per_device_train_batch_size}")
print(f"📋 Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"📋 Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📋 Epochs: {training_args.num_train_epochs}")
print(f"📋 LR: {training_args.learning_rate}")
print(f"📋 FP16: {training_args.fp16}")

In [ ]:
# ── Start training ─────────────────────────────────────────────────────
print("🚀 Starting fine-tuning...")

# Qwen2.5 doesn't need 4-bit for 1.5B, but we use it for memory safety
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    max_seq_length=1024,
    dataset_text_field="text",
)

trainer.train()
print("\n✅ Training complete!")

In [ ]:
# ── Save LoRA adapter ──────────────────────────────────────────────────
save_path = "/content/tg-lora-adapter"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ LoRA adapter saved to {save_path}")

import shutil
shutil.make_archive("/content/tg-lora-adapter", "zip", save_path)
print(f"✅ Zipped: /content/tg-lora-adapter.zip")

In [ ]:
# ── Merge + export to GGUF (for Ollama) ────────────────────────────────
# Step 1: Merge LoRA weights into base model
print("🔄 Merging LoRA weights...")

from peft import PeftModel

# Reload base model without quantization
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",  # merge on CPU to avoid OOM
    trust_remote_code=True,
)

merged = PeftModel.from_pretrained(base, save_path)
merged = merged.merge_and_unload()

merged_save = "/content/tg-merged"
merged.save_pretrained(merged_save)
tokenizer.save_pretrained(merged_save)
print(f"✅ Merged model saved to {merged_save}")

In [ ]:
# ── Convert to GGUF ────────────────────────────────────────────────────
print("🔄 Converting to GGUF (Q4_K_M quantization)...")

# Install llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && make -j2 convert-hf-to-gguf 2>&1 | tail -5

# Install Python deps for conversion script
!pip install -q /content/llama.cpp[convert] 2>&1 | tail -3

# Run conversion
!python /content/llama.cpp/convert_hf_to_gguf.py /content/tg-merged \
    --outfile /content/tg-tutor.gguf \
    --outtype q4_k_m 2>&1 | tail -10

import os
gguf_size = os.path.getsize("/content/tg-tutor.gguf")
print(f"\n✅ GGUF created: {gguf_size/1024/1024:.0f} MB")

In [ ]:
# ── Download GGUF + Modelfile ───────────────────────────────────────────
from google.colab import files

# Create Ollama Modelfile
modelfile_content = f"""FROM /content/tg-tutor.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
{{ .Response }}<|im_end|"""

SYSTEM "أنت مساعد تربوي ذكي للأهل العرب المسلمين. أجب على آخر سؤال فقط."

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
"""

with open("/content/Modelfile.tg-tutor", "w") as f:
    f.write(modelfile_content)

print("📥 Downloading GGUF (≈1.5 GB for 3B Q4)...")
files.download("/content/tg-tutor.gguf")
files.download("/content/Modelfile.tg-tutor")
print("✅ Downloads initiated")

In [ ]:
# ── Quick test before downloading ──────────────────────────────────────
print("🧪 Testing fine-tuned model...")

import subprocess

# Run a quick Ollama-style prompt through llama.cpp
test_prompt = "<|im_start|>system\nأنت مساعد تربوي ذكي.\n<|im_end|>\n<|im_start|>user\nكيف أتعامل مع ابني الذي يغضب كثيراً؟<|im_end|>\n<|im_start|>assistant\n"

# Use llama.cpp's main to test
!echo "{test_prompt}" | /content/llama.cpp/main \
    -m /content/tg-tutor.gguf \
    -n 200 \
    --temp 0.3 \
    --repeat-penalty 1.1 \
    -p "$(cat)" 2>/dev/null

print("\n---\n✅ If you see a coherent Arabic response, the model works!")